This is the main tutorial notebook. It is intended to introduce the basic machinery and how it is used.

First, let's import everything we need.

In [133]:
# import packages
%matplotlib inline
import numpy as np
import math
import sys
# sys.path.append('.')  # or full path to CompSep
np.math = math  # Redirect numpy.math to the built-in math module
import torch
import denoising
import importlib
denoising = importlib.reload(denoising)
import utils
utils = importlib.reload(utils)

Here we import the data. Since this is simulation data of log column density (in units of $1/cm^3$). If the data is already preprocessed, you can just import it as is.

In [134]:
def create_apodizing_mask_np(M, N, J):
    """
    Create an apodizing mask of shape (1, M, N) using NumPy.
    """
    margin = 2**(J - 3)
    mask = np.zeros((M, N), dtype=np.float32)
    mask[margin:M - margin, margin:N - margin] = 1.0
    return mask[None, :, :]  # shape (1, M, N)

In [135]:
# Load the column density map from file
logN_H = np.load('Archive/Turb_3.npy')[1]  
# logN_H = utils.downsample_by_four(logN_H)
nx, ny = logN_H.shape

dx, dy = nx // 4, ny // 4
logN_H = logN_H[nx//2 - dx : nx//2 + dx, ny//2 - dy : ny//2 + dy]

# Define constant dust temperature
T_d = 10  # Typical Planck dust temperature in K

# int_val = []
# for nu in nu_list:
#     int_val.append(np.mean(modified_blackbody(logN_H, T_d, nu*1e9)))
# int_val = np.array(int_val)

# Observation frequency (e.g., 353 GHz in Hz)

# Compute the mock observed intensity map in μK_CMB
nu = (217,353)
# I_nu_map_μK_nu1 = utils.modified_blackbody(logN_H, T_d, nu[0]*1e9)
# I_nu_map_μK_nu2 = utils.modified_blackbody(logN_H, T_d, nu[1]*1e9)
alpha_1 = utils.MBB_factor(T_d, nu[0]*1e9)
alpha_2 = utils.MBB_factor(T_d, nu[1]*1e9)
I_nu_map_μK_nu1 = 10**logN_H*alpha_1
I_nu_map_μK_nu2 = 10**logN_H*alpha_2

I_nu_map_μK = (I_nu_map_μK_nu1, I_nu_map_μK_nu2)

The arrays that we work with need to have dimension `(1, H, W)`.

Now we define tuples of dust which will be our ground truth $s$, white noise, $n$, and finally cmb $c$. For simplicity the CMB here is modelled as a simple Gaussian random field with a falling power law for the power spectrum.

Keep in mind that in general, you can just import your own contamination array, respecting the broadcasting.

In [136]:
M,N = I_nu_map_μK_nu1.shape
J = int(np.log2(M)) - 1
mask = create_apodizing_mask_np(M, N, J)


# Ensure inputs have shape (1, H, W)
dust_nu1 = I_nu_map_μK_nu1[None, :, :]
dust_nu2 = I_nu_map_μK_nu2[None, :, :]

dust = (dust_nu1, dust_nu2)

# --- CMB Parameters ---
n_realizations = 100
SNR = 2
amplitude = 0.5
# spectral_index = -1.7
spectral_index = -1.2
nx, ny = dust_nu1.shape[-2], dust_nu1.shape[-1]

# --- Compute noise variances ---
variance_nu1 = (np.std(dust_nu1) / SNR) ** 2
variance_nu2 = (np.std(dust_nu2) / SNR) ** 2
variance = (variance_nu1, variance_nu2)

# --- Create contamination_arr with shape (n_realizations, 2, 1, H, W) ---
contamination_arr = np.zeros((n_realizations, 2, 1, nx, ny), dtype=np.float32)

for i in range(n_realizations):
    # Shared CMB: shape (1, H, W)
    cmb_map = utils.generate_cmb_map(n_x=nx, n_y=ny, amplitude=amplitude, spectral_index=spectral_index)
    cmb_map = cmb_map.cpu().numpy()[None, :, :]

    # Independent noise: shape (1, H, W)
    noise_nu1 = np.random.normal(0, np.sqrt(variance_nu1), (1, nx, ny)) * mask
    noise_nu2 = np.random.normal(0, np.sqrt(variance_nu2), (1, nx, ny)) * mask

    # Total contamination: shape (1, H, W)
    contamination_arr[i, 0] = noise_nu1 #+ cmb_map
    contamination_arr[i, 1] = noise_nu2 #+ cmb_map

contamination_arr_nu1 = contamination_arr[:, 0]  # shape: (Mn, 1, H, W)
contamination_arr_nu2 = contamination_arr[:, 1]  # shape: (Mn, 1, H, W)

Here we create our data, given by the equation $d^\nu = s^\nu + c + n^\nu$ for each frequency band $\nu$. Notice that we are assuming that we can sample configures $c \sim P(c)$ and $n \sim P(n)$. While for this case this might be possible, in general for more complicated contaminations this may be more difficult. This is a strong assumption.

We assume that $c$ is independent of $\nu$ because its spectral energy function is that of a perfect black body, and therefore $c^\nu = B(\nu, T) c_0$, where $c_0$ is some configuration and all the frequency dependce factors out into the Planck function. Therefore, we can define units (called Kelvin CMB) where this is independent of frequency.

The tuple contains the data for two channels, that is, $(d^{\nu_1}, d^{\nu_2})$.

In [137]:
noise_nu1 = np.random.normal(0, np.sqrt(variance_nu1), dust_nu1.shape)
noise_nu2 = np.random.normal(0, np.sqrt(variance_nu2), dust_nu2.shape)

noise = (noise_nu1, noise_nu2)

cmb_map = utils.generate_cmb_map(n_x=nx, n_y=ny, amplitude=amplitude, spectral_index=spectral_index)
cmb_map = cmb_map.cpu().numpy()[None, :, :]

data_nu1 = dust_nu1 + noise_nu1
data_nu2 = dust_nu2 + noise_nu2

# define target
data = (data_nu1, data_nu2)
data = tuple(img * mask for img in data)

# target image is the data
image_target = data

# definte initial maps for optimisation
image_init = None
# image_init = ( 1e-20*data[0] / alpha_1,)
# image_init = (np.random.normal(0, 1e-5, size=data[0].shape).astype(np.float32),)

If $x$ is an image, then $\phi(x)$ denotes a set of summary statistics. For example, if can be the pixel average, and the pixel wide standard deviation. In that case, it would be $\phi(x) = ( \mu, \sigma )$. We are going to work with a different kind of coefficients, called Scattering Covariances

In [138]:
thresholding = False
if thresholding:
    M, N, J, L = dust_nu1.shape[-2], dust_nu1.shape[-1], 7, 4
    st_calc = denoising.Scattering2d(M, N, J, L)
    st_calc.add_ref(ref=data_nu1)
    s_cov = st_calc.scattering_cov(data_nu1, use_ref=True, 
                        normalization='P00', pseudo_coef=1
                    )
    st_calc.add_ref_ab(ref_a=data_nu1, ref_b=data_nu2)
    s_cov_2fields = st_calc.scattering_cov_2fields(data_nu1, data_nu2, use_ref=True, 
                        normalization='P00'
                    )
    threshold_func = denoising.threshold_func(s_cov)
    threshold_func_2fields = denoising.threshold_func(s_cov_2fields, two_fields=True)

else:
    threshold_func = None
    threshold_func_2fields = None

In [ ]:
image_init = image_target
threshold_func = None
remove_edge = False

std = {
    'single': denoising.compute_std(image_init, contamination_arr=contamination_arr,
                                    s_cov_func=threshold_func, remove_edge=remove_edge, precision='single')

    'double': denoising.compute_std_double(image_init, contamination_arr=contamination_arr,
                                           remove_edge=remove_edge, precision='single')
}

In [ ]:
n_epochs = 3 #number of epochs
# decontaminate
for i in range(n_epochs):
    print(f'Starting epoch {i+1}')
    running_map = denoising.denoise(image_target, contamination_arr = contamination_arr, std = std, seed=0, print_each_step=True, fixed_img= None,
                                    steps = 25, n_batch = 25, s_cov_func=threshold_func, image_init = image_init, remove_edge=remove_edge, precision='single', 
                                    if_large_batch=False, epochNo = i)
    running_map = (running_map[0], running_map[1])
    # image_init = running_map


    std = {
        'single': denoising.compute_std(running_map, contamination_arr=contamination_arr,
                                        s_cov_func=threshold_func, remove_edge=remove_edge, precision='single'), 

        'double': denoising.compute_std_double(running_map, contamination_arr=contamination_arr,
                                           remove_edge=remove_edge, precision='single')
    }
    image_init = image_target
    
image_syn_Q = running_map[0]
image_syn_U = running_map[1]


Starting epoch 1
Current Loss: 1.45e+01
Current Loss: 1.45e+01
Current Loss: 1.45e+01
Current Loss: 1.44e+01
Current Loss: 1.42e+01
Current Loss: 1.07e+01
Current Loss: 8.87e+00
Current Loss: 6.81e+00
Current Loss: 5.69e+00
Current Loss: 4.50e+00
Current Loss: 3.71e+00
Current Loss: 3.14e+00
Current Loss: 2.62e+00
Current Loss: 2.39e+00
Current Loss: 2.12e+00
Current Loss: 1.97e+00
Current Loss: 1.82e+00
Current Loss: 1.72e+00
Current Loss: 1.65e+00
Current Loss: 1.63e+00
Current Loss: 1.57e+00
Current Loss: 1.55e+00
Current Loss: 1.56e+00
Current Loss: 1.51e+00
Current Loss: 1.51e+00
Time used:  61.055006980895996 s
Starting epoch 2
Current Loss: 7.29e+00
Current Loss: 7.30e+00
Current Loss: 7.31e+00
Current Loss: 7.27e+00
Current Loss: 7.17e+00
Current Loss: 5.69e+00
Current Loss: 4.89e+00
Current Loss: 4.03e+00
Current Loss: 3.50e+00
Current Loss: 2.92e+00
Current Loss: 2.55e+00
Current Loss: 2.22e+00
Current Loss: 1.93e+00
Current Loss: 1.82e+00
Current Loss: 1.65e+00
Current Loss:

In [45]:
# Convert tuples to NumPy arrays
dust = np.stack([dust_nu1[0], dust_nu2[0]])  # Shape: (2, ...)
data = np.stack([data_nu1[0], data_nu2[0]])  # Shape: (2, ...)
image_denoised = np.stack([image_syn_nu1[0], image_syn_nu2[0]])  # Ensure it's an array

cmb = True
# Create an array of objects to preserve different shapes
results = np.array([dust, data, image_denoised])
# np.save(f"nu={nu}_cmb={cmb}_edge={not remove_edge}", results)